# CSV -> Delta (bulk)

Draai dit notebook in de Fabric-workspace, met de Lakehouse `Landing_bron_data`
als **default lakehouse** gekoppeld aan het notebook (linkerpaneel -> Lakehouses
toevoegen -> als default instellen).

Zet alle `.csv`-bestanden uit `Files/` (of de submap die je in `source_folder`
opgeeft) in één run om naar Delta-tabellen in `Tables/dbo/`, i.p.v. losse
"Load to Tables" acties per bestand. Idempotent - opnieuw draaien overschrijft
gewoon dezelfde tabellen, dus veilig om te herhalen als er nieuwe CSV's bijkomen.

In [ ]:
# Parameters - pas aan indien nodig
source_folder = "Files"     # of bv. "Files/landing" als de CSV's in een submap staan
target_schema = "dbo"
csv_delimiter = ","

In [ ]:
def list_csv_files(path):
    result = []
    for item in notebookutils.fs.ls(path):
        if item.isDir:
            result.extend(list_csv_files(item.path))
        elif item.name.lower().endswith(".csv"):
            result.append(item)
    return result

csv_files = list_csv_files(source_folder)
print(f"Gevonden: {len(csv_files)} CSV-bestanden")
for f in csv_files:
    print(" -", f.name)

In [ ]:
loaded = []
failed = []

for f in csv_files:
    table_name = f.name[:-4]  # strip ".csv"
    try:
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("delimiter", csv_delimiter)
            .csv(f.path)
        )
        df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{target_schema}.{table_name}")
        loaded.append(table_name)
        print(f"OK   - {table_name} ({df.count()} rijen)")
    except Exception as e:
        failed.append((table_name, str(e)))
        print(f"FOUT - {table_name}: {e}")

print(f"\nKlaar: {len(loaded)} geladen, {len(failed)} gefaald.")
if failed:
    raise RuntimeError(f"Mislukte tabellen: {[name for name, _ in failed]}")